# Fine Tuning

In [ ]:
!pip install gensim
!pip install --upgrade datasets==3.6.0

In [ ]:
import os
import math
import random
import json
import pickle
import re
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from openai import OpenAI
from datetime import datetime
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from huggingface_hub import login
from sklearn.svm import LinearSVR
from gensim.models import Word2Vec
from IPython.display import display
from transformers import AutoTokenizer
from gensim.utils import simple_preprocess
from collections import Counter, defaultdict
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from concurrent.futures import ProcessPoolExecutor
from datasets import Dataset, DatasetDict, load_dataset
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
load_dotenv(override=True)
api_key_openai = os.environ.get("OPENAI_API_KEY")

hub_token = os.environ.get("HF_TOKEN")
print(hub_token)

if hub_token:
    print("Loggin in...")
    login(hub_token, add_to_git_credential=True)

In [ ]:
COLOR_GREEN = "\033[92m"
COLOR_YELLOW = "\033[93m"
COLOR_RED = "\033[91m"
COLOR_RESET = "\033[0m"
ERROR_COLOR = {"red": COLOR_RED, "orange": COLOR_YELLOW, "green": COLOR_GREEN}

In [ ]:
FOUNDATION_MODEL = "meta-llama/Meta-Llama-3.1-8B"

MINIMUM_CHARS = 300
MINIMUM_TOKENS = 150
TOKEN_LIMIT = 160
CHAR_LIMIT = TOKEN_LIMIT * 7

class ProductSample:
    tokenizer = AutoTokenizer.from_pretrained(FOUNDATION_MODEL, trust_remote_code=True)
    PRICE_PREFIX = "Price is $"
    PRICE_QUESTION = "How much does this cost to the nearest dollar?"
    STRIP_PHRASES = ['"Batteries Included?": "No"', '"Batteries Included?": "Yes"', '"Batteries Required?": "No"', '"Batteries Required?": "Yes"', "By Manufacturer", "Item", "Date First", "Package", ":", "Number of", "Best Sellers", "Number", "Product "]

    def __init__(self, record, price):
        self.title = record["title"]
        self.price = price
        self.category = record.get("category", "Unknown")
        self.token_count = 0
        self.details = None
        self.prompt = None
        self.include = False
        self._parse(record)

    def _scrub_details(self):
        text = self.details
        for phrase in self.STRIP_PHRASES:
            text = text.replace(phrase, "")
        return text

    def _scrub_text(self, text):
        text = re.sub(r'[:\[\]"{}【】\s]+', ' ', text).strip()
        text = text.replace(" ,", ",").replace(",,,",",").replace(",,",",")
        tokens = text.split(" ")
        kept = [t for t in tokens if len(t) < 7 or not any(ch.isdigit() for ch in t)]
        return " ".join(kept)

    def _parse(self, record):
        body = '\n'.join(record.get("description", []))
        if body:
            body += '\n'

        feats = '\n'.join(record.get("features", []))
        if feats:
            body += feats + '\n'

        self.details = record.get("details")
        if self.details:
            body += self._scrub_details() + '\n'

        if len(body) > MINIMUM_CHARS:
            body = body[:CHAR_LIMIT]
            text = f"{self._scrub_text(self.title)}\n{self._scrub_text(body)}"
            ids = self.tokenizer.encode(text, add_special_tokens=False)

            if len(ids) > MINIMUM_TOKENS:
                ids = ids[:TOKEN_LIMIT]
                text = self.tokenizer.decode(ids)
                self._make_prompt(text)
                self.include = True

    def _make_prompt(self, text):
        self.prompt = f"{self.PRICE_QUESTION}\n\n{text}\n\n"
        self.prompt += f"{self.PRICE_PREFIX}{str(round(self.price))}.00"
        self.token_count = len(self.tokenizer.encode(self.prompt, add_special_tokens=False))

    def test_prompt(self):
        return self.prompt.split(self.PRICE_PREFIX)[0] + self.PRICE_PREFIX

    def __repr__(self):
        return f"<{self.title} = ${self.price}>"

In [ ]:
MIN_ALLOWED_PRICE = 0.5
BATCH_SIZE = 1000
MAX_ALLOWED_PRICE = 999.49

class CatalogLoader:
    def __init__(self, catalog_name):
        self.catalog_name = catalog_name
        self.dataset = None

    def _from_row(self, row):
        try:
            raw = row.get("price")
            if raw:
                value = float(raw)
                if MIN_ALLOWED_PRICE <= value <= MAX_ALLOWED_PRICE:
                    sample = ProductSample(row, value)
                    if sample.include:
                        return sample
        except ValueError:
            return None

    def _from_batch(self, batch):
        out = []
        for row in batch:
            s = self._from_row(row)
            if s:
                out.append(s)
        return out

    def _batches(self):
        n = len(self.dataset)
        for start in range(0, n, BATCH_SIZE):
            yield self.dataset.select(range(start, min(start + BATCH_SIZE, n)))

    def load_parallel(self, workers):
        items = []
        batch_count = (len(self.dataset) // BATCH_SIZE) + 1

        with ProcessPoolExecutor(max_workers=workers) as pool:
            for batch in tqdm(pool.map(self._from_batch, self._batches()), total=batch_count):
                items.extend(batch)

        for it in items:
            it.category = self.catalog_name

        return items

    def load(self, workers=8):
        self.dataset = load_dataset(
            "McAuley-Lab/Amazon-Reviews-2023",
            f"raw_meta_{self.catalog_name}",
            split="full",
            trust_remote_code=True,
        )
        start = datetime.now()
        print(f"Loading {self.dataset}")
        items = self.load_parallel(workers)
        mins = (datetime.now() - start).total_seconds() / 60
        print(f"Completed {self.catalog_name} with {len(items):,} items in {mins:.1f} mins")
        return items

In [ ]:
class Evaluator:

    def __init__(self, predictor, data, title=None, size=250):
        self.predictor = predictor
        self.data = data
        self.title = title or predictor.__name__.replace("_", " ").title()
        self.size = size
        self.preds = []
        self.labels = []
        self.abs_errors = []
        self.sle_values = []
        self.colors = []

    def _pick_color(self, error, truth):
        if error < 40 or error / truth < 0.2:
            return "green"
        if error < 80 or error / truth < 0.4:
            return "orange"
        return "red"

    def _run_one(self, idx):
        row = self.data[idx]
        pred = self.predictor(row)
        truth = row.price
        err = abs(pred - truth)
        log_err = math.log(truth + 1) - math.log(pred + 1)
        sle = log_err ** 2
        color = self._pick_color(err, truth)
        name = row.title if len(row.title) <= 40 else row.title[:40] + "..."

        self.preds.append(pred)
        self.labels.append(truth)
        self.abs_errors.append(err)
        self.sle_values.append(sle)
        self.colors.append(color)

        print(
            f"{ERROR_COLOR[color]}{idx + 1}: Guess: ${pred:,.2f} Truth: ${truth:,.2f} "
            f"Error: ${err:,.2f} SLE: {sle:,.2f} Item: {name}{COLOR_RESET}"
        )

    def _plot(self, title):
        plt.figure(figsize=(12, 8))
        max_val = max(max(self.labels), max(self.preds))
        plt.plot([0, max_val], [0, max_val], color="deepskyblue", lw=2, alpha=0.6)
        plt.scatter(self.labels, self.preds, s=3, c=self.colors)
        plt.xlabel("Ground Truth")
        plt.ylabel("Model Estimate")
        plt.xlim(0, max_val)
        plt.ylim(0, max_val)
        plt.title(title)
        plt.show()

    def report(self):
        mean_abs_error = sum(self.abs_errors) / self.size
        rmsle = math.sqrt(sum(self.sle_values) / self.size)
        hit_rate = sum(1 for c in self.colors if c == "green") / self.size
        title = f"{self.title} Error=${mean_abs_error:,.2f} RMSLE={rmsle:,.2f} Hits={hit_rate * 100:.1f}%"
        self._plot(title)

    def run(self):
        for idx in range(self.size):
            self._run_one(idx)
        self.report()

    @classmethod
    def evaluate(cls, fn, data):
        cls(fn, data).run()

def parse_price(text):
    text = text.replace("$", "").replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+", text)
    return float(m.group()) if m else 0.0

## Data

### Load Catalogs

In [ ]:
catalog_names = [
    "All_Beauty",
    # "Automotive",
    # "Electronics",
    # "Office_Products",
    # "Tools_and_Home_Improvement",
    # "Cell_Phones_and_Accessories",
    # "Toys_and_Games",
    "Appliances",
    "Musical_Instruments",
    "Software",
    "Handmade_Products",
]

sample_pool = []

for name in catalog_names:
    print("Loading " + name)
    loader = CatalogLoader(name)
    sample_pool.extend(loader.load())

print(f"Total curated items: {len(sample_pool):,}")

In [ ]:
prices_vec = [s.price for s in sample_pool]
tokens_vec = [s.token_count for s in sample_pool]
category_counts = Counter(s.category for s in sample_pool)
summary_df = pd.DataFrame({"price": prices_vec, "tokens": tokens_vec})

display(summary_df.describe())
display(pd.DataFrame.from_dict(category_counts, orient="index", columns=["count"]).sort_values("count", ascending=False))

In [ ]:
buckets = defaultdict(list)
for s in sample_pool:
    k = round(s.price)
    if 1 <= k <= 999:
        buckets[k].append(s)

bucket_sizes = {k: len(v) for k, v in buckets.items()}
print(f"Slots populated: {len(bucket_sizes)}")

In [ ]:
random.seed(123)
np.random.seed(123)

balanced_pool = []

for price in range(1, 1000):
    bucket = buckets.get(price, [])

    if price >= 240:
        balanced_pool.extend(bucket)
    elif len(bucket) <= 1200:
        balanced_pool.extend(bucket)
    else:
        # Same sampling logic; slight refactor for readability
        weights = np.array([1 if s.category == "Automotive" else 5 for s in bucket], dtype=float)
        weights = weights / weights.sum()
        picks = np.random.choice(len(bucket), size=1200, replace=False, p=weights)
        balanced_pool.extend([bucket[i] for i in picks])

print(f"Balanced bundle size: {len(balanced_pool):,}")

In [ ]:
bal_prices = [s.price for s in balanced_pool]
bal_tokens = [s.token_count for s in balanced_pool]
bal_cat = Counter(s.category for s in balanced_pool)

display(pd.Series(bal_prices).describe())
display(pd.DataFrame.from_dict(bal_cat, orient="index", columns=["count"]).sort_values("count", ascending=False))

In [ ]:
plt.figure(figsize=(12, 5))
plt.hist(bal_prices, bins=range(0, 1000, 10), color="midnightblue", rwidth=0.8)
plt.xlabel("Price")
plt.ylabel("Count")

plt.figure(figsize=(12, 5))
plt.hist(bal_tokens, bins=range(0, 300, 10), color="forestgreen", rwidth=0.8)
plt.xlabel("Tokens")
plt.ylabel("Count")
plt.show()

In [ ]:
random.seed(123)
random.shuffle(balanced_pool)

test_n = min(2000, max(1, len(balanced_pool) // 20))
train_n = min(400_000, len(balanced_pool) - test_n)

train_samples = balanced_pool[:train_n]
test_samples = balanced_pool[train_n:train_n + test_n]

print(f"Training set: {len(train_samples):,}")
print(f"Test set: {len(test_samples):,}")

In [ ]:
train_texts = [s.prompt for s in train_samples]
train_labels = [s.price for s in train_samples]

test_texts = [s.test_prompt() for s in test_samples]
test_labels = [s.price for s in test_samples]

In [ ]:
train_ds = Dataset.from_dict({"text": train_texts, "price": train_labels})
test_ds = Dataset.from_dict({"text": test_texts, "price": test_labels})
price_ds = DatasetDict({"train": train_ds, "test": test_ds})

### Persist

In [ ]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "balanced_train.pkl", "wb") as f:
    pickle.dump(train_samples, f)

with open(data_dir / "balanced_test.pkl", "wb") as f:
    pickle.dump(test_samples, f)

In [ ]:
price_ds["train"].to_parquet(data_dir / "balanced_train.parquet")
price_ds["test"].to_parquet(data_dir / "balanced_test.parquet")

## Baselines

### Stochastic Anchor

In [ ]:
def random_anchor(_sample):
    return random.randrange(1, 1000)

random.seed(123)
Evaluator.evaluate(random_anchor, test_samples[:250])

### Global Mean

In [ ]:
train_price_list = [s.price for s in train_samples]
mean_price = sum(train_price_list) / len(train_price_list)

def mean_baseline(_sample):
    return mean_price

Evaluator.evaluate(mean_baseline, test_samples[:250])

In [ ]:
def try_parse_features(raw):
    if not raw:
        return {}
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {}

for s in train_samples:
    s.features = try_parse_features(s.details)
for s in test_samples:
    s.features = try_parse_features(s.details)

### Feature Engineering

In [ ]:
def extract_weight(sample):
    payload = sample.features.get("Item Weight")
    if not payload:
        return None

    parts = payload.split(" ")
    amount = float(parts[0])
    unit = parts[1].lower()

    if unit == "pounds":
        return amount
    if unit == "ounces":
        return amount / 16
    if unit == "grams":
        return amount / 453.592
    if unit == "milligrams":
        return amount / 453592
    if unit == "kilograms":
        return amount / 0.453592
    if unit == "hundredths" and len(parts) > 2 and parts[2].lower() == "pounds":
        return amount / 100

    return None

In [ ]:
def extract_rank(sample):
    payload = sample.features.get("Best Sellers Rank")
    if not payload:
        return None

    values = list(payload.values()) if isinstance(payload, dict) else []
    if not values:
        return None

    return sum(values) / len(values)

KNOWN_TOP_BRANDS = {"nvidea", "hp", "dell", "lenovo", "samsung", "asus", "sony", "canon", "apple", "intel"}

def is_known_brand(sample):
    brand = sample.features.get("Brand")
    return 1 if brand and brand.lower() in KNOWN_TOP_BRANDS else 0

In [ ]:
weights = [extract_weight(s) for s in train_samples]
weights = [w for w in weights if w is not None]
mean_weight = sum(weights) / len(weights) if weights else 1.0

ranks = [extract_rank(s) for s in train_samples]
ranks = [r for r in ranks if r is not None]
mean_rank = sum(ranks) / len(ranks) if ranks else 1_000_000.0

In [ ]:
def features_for(sample):
    w = extract_weight(sample)
    r = extract_rank(sample)
    return {
        "weight": w if w is not None else mean_weight,
        "rank": r if r is not None else mean_rank,
        "text_length": len(sample.test_prompt()),
        "top_brand": is_known_brand(sample),
    }

In [ ]:
train_features_df = pd.DataFrame([features_for(s) for s in train_samples])
train_features_df["price"] = [s.price for s in train_samples]

test_features_df = pd.DataFrame([features_for(s) for s in test_samples[:250]])
test_features_df["price"] = [s.price for s in test_samples[:250]]

In [ ]:
feature_cols = ["weight", "rank", "text_length", "top_brand"]
X_train = train_features_df[feature_cols]
y_train = train_features_df["price"]

X_test = test_features_df[feature_cols]
y_test = test_features_df["price"]

regression = LinearRegression()
regression.fit(X_train, y_train)

def linear_predict(sample):
    return float(regression.predict(pd.DataFrame([features_for(sample)]))[0])

Evaluator.evaluate(linear_predict, test_samples[:250])

### NLP Baselines

In [ ]:
train_docs = [s.test_prompt() for s in train_samples]
y_targets = np.array([s.price for s in train_samples])

bow = CountVectorizer(max_features=1000, stop_words="english")
X_bow = bow.fit_transform(train_docs)

bow_reg = LinearRegression()
bow_reg.fit(X_bow, y_targets)

def bow_predict(sample):
    pred = float(bow_reg.predict(bow.transform([sample.test_prompt()]))[0])
    return max(pred, 0)

Evaluator.evaluate(bow_predict, test_samples[:250])

In [ ]:
tokenized = [simple_preprocess(t) for t in train_docs]
w2v = Word2Vec(sentences=tokenized, vector_size=400, window=5, min_count=1, workers=4)

def vectorize(text):
    words = simple_preprocess(text)
    vecs = [w2v.wv[w] for w in words if w in w2v.wv]
    if not vecs:
        return np.zeros(w2v.vector_size)
    return np.mean(vecs, axis=0)

X_w2v = np.array([vectorize(t) for t in train_docs])
svr = LinearSVR()
svr.fit(X_w2v, y_targets)

def w2v_predict(sample):
    return float(svr.predict([vectorize(sample.test_prompt())])[0])

Evaluator.evaluate(w2v_predict, test_samples[:250])

In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=123)
rf.fit(X_train, y_train)

def rf_predict(sample):
    return float(rf.predict(pd.DataFrame([features_for(sample)])[feature_cols])[0])

Evaluator.evaluate(rf_predict, test_samples[:250])

In [ ]:
ft_train = train_samples[:200]
ft_valid = train_samples[200:250]

In [ ]:
def build_messages(sample, include_price=True):
    system_text = "You estimate prices of items. Reply only with the price"
    user_text = sample.test_prompt().replace(" to the nearest dollar", "").replace("\n\nPrice is $", "")
    assistant_text = f"Price is ${sample.price:.2f}" if include_price else "Price is $"
    return [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": assistant_text},
    ]

In [ ]:
def to_jsonl(samples):
    rows = []
    for s in samples:
        rows.append(json.dumps({"messages": build_messages(s)}))
    return "\n".join(rows)

In [ ]:
train_path = data_dir / "balanced_pricer_train.jsonl"
valid_path = data_dir / "balanced_pricer_validation.jsonl"

train_path.write_text(to_jsonl(ft_train))
valid_path.write_text(to_jsonl(ft_valid))

In [ ]:
client = OpenAI()

with open(train_path, "rb") as f:
    train_file = client.files.create(file=f, purpose="fine-tune")

with open(valid_path, "rb") as f:
    valid_file = client.files.create(file=f, purpose="fine-tune")

train_file, valid_file

In [ ]:
wandb_cfg = {"type": "wandb", "wandb": {"project": "balanced-pricer"}}
job = client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=valid_file.id,
    model="gpt-4o-mini-2024-07-18",
    seed=123,
    hyperparameters={"n_epochs": 1},
    integrations=[wandb_cfg],
    suffix="balanced-pricer",
)
job

In [ ]:
status = client.fine_tuning.jobs.retrieve(job.id)
events = client.fine_tuning.jobs.list_events(fine_tuning_job_id=job.id, limit=10)
status, events

In [ ]:
model_name = client.fine_tuning.jobs.retrieve(job.id).fine_tuned_model
print(model_name)

In [ ]:
def tuned_price_predict(sample):
    msgs = build_messages(sample, include_price=False)
    resp = client.chat.completions.create(
        model=model_name,
        messages=msgs,
        seed=123,
        max_tokens=7,
    )
    text = resp.choices[0].message.content
    return parse_price(text)

In [ ]:
if test_samples:
    ex = test_samples[0]
    print(ex.price)
    print(tuned_price_predict(ex))

In [ ]:
Evaluator.evaluate(tuned_price_predict, test_samples[:250])